# C_01 — Small impacts and global-melting baseline (sulfur workflow)

Read the collision histories from `B_02_accretion_history_s.ipynb`. `ImpactEventProcessor` calculates small-impact equilibration conditions and provisionally identifies global magma-ocean events. This first pass is required before `C_02` updates qualifying giant impacts to partial melting.

**Input:** `Tables/04_accretion_history_S.xlsx`  
**Output:** `Tables/05_small_global_impact_events_S.xlsx` (intermediate, not input to D_01)

Run from the project root or `notebooks/work_flow_S`.


In [1]:
# Small impacts and global-melting baseline.
PROJECT_ROOT = None
WORKFLOW_DIR = None
ACCRETION_FILE = None
P_RATIO = 0.6                 # Small-impact equilibration-pressure fraction
GLOBAL_MASS_THRESHOLD = 0.1 # M Earth
IMPACT_RATIO_RANGE = (0.1, 0.5)
COOLING_DURATION_MYR = 5.0
OUTPUT_EVENTS_FILE = None
OUTPUT_PARAMETERS_FILE = None


In [2]:
import matplotlib.pyplot as plt
import json
import numpy as np
import pandas as pd
import sys
from matplotlib.lines import Line2D
from pathlib import Path

working_dir = Path.cwd()
project_root = (Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT is not None
                else next((p for p in (working_dir, *working_dir.parents)
                           if (p / "src" / "accrediff").is_dir()), None))
if project_root is None:
    raise RuntimeError("Could not locate the AccreDiff repository root.")
workflow_dir = (Path(WORKFLOW_DIR).expanduser().resolve() if WORKFLOW_DIR is not None
                else project_root / 'kong_2027_backup' / 'work_flow_S')

sys.path.insert(0, str(project_root / 'src'))
import accrediff as ad


# ── Plot style ──────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 1,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
    "figure.dpi": 150,
})

print(f"Project root : {project_root}")
print("Setup complete.")


Project root : /Users/kong/Research/code_zju/AccreDiff
Setup complete.


## 1. Load collision histories


In [3]:
# ── Load and validate accretion histories ────────────────
accretion_file = Path(ACCRETION_FILE) if ACCRETION_FILE is not None else workflow_dir / 'Tables' / '04_accretion_history_S.xlsx'
if not accretion_file.is_file():
    raise FileNotFoundError(
        f'Accretion history not found: {accretion_file}. Run B_02_accretion_history_s.ipynb first.'
    )

D = pd.read_excel(accretion_file, sheet_name=None)
if not D:
    raise ValueError(f'No planet sheets found in {accretion_file}.')

required_history_columns = {
    'Time', 'indexi', 'm_ei', 'indexj', 'm_ej', 'product_id', 'Mass'
}
for key, frame in D.items():
    missing = required_history_columns - set(frame.columns)
    if missing:
        raise ValueError(f'{key} lacks required collision columns: {sorted(missing)}')
    if frame.empty:
        raise ValueError(f'{key} contains no post-disk collision events.')

print(f"Loaded  : {accretion_file}")
print(f"Planets : {list(D)}")


Loaded  : /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S/Tables/04_accretion_history_S.xlsx
Planets : ['planets_1', 'planets_2', 'planets_3', 'planets_4', 'planets_5', 'planets_6']


## 2. Calculate the small-impact and provisional global conditions

`P_RATIO` controls local small-impact equilibration pressure. The mass threshold, impactor ratio, and cooling duration determine the provisional global-melting history. Global properties must be calculated even for events later reclassified as `partial`.


In [4]:
if not (0.0 < P_RATIO <= 1.0 and GLOBAL_MASS_THRESHOLD > 0.0 and COOLING_DURATION_MYR > 0.0):
    raise ValueError('Invalid C_01 impact parameters.')
if len(IMPACT_RATIO_RANGE) != 2 or not 0.0 <= IMPACT_RATIO_RANGE[0] < IMPACT_RATIO_RANGE[1] <= 1.0:
    raise ValueError('Invalid impact-ratio range.')

data_dict = {}

for key, collision_history in D.items():
    processor = ad.ImpactEventProcessor(
        collision_history.copy(),
        p_ratio=P_RATIO,
        Mtotal=GLOBAL_MASS_THRESHOLD,
        ratio=list(IMPACT_RATIO_RANGE),
    )
    data_dict[key] = processor.update_global_melting_process(
        duration=COOLING_DURATION_MYR
    )

first_planet = next(iter(data_dict))
print("Global melting classification complete.")
print(f"Event type counts ({first_planet}):")
print(data_dict[first_planet]['events'].value_counts())


Global melting classification complete.
Event type counts (planets_1):
events
small     115
global     12
Name: count, dtype: int64


## 3. Validate and export the intermediate event table


In [5]:
required_event_columns = {
    'Time', 'target_id', 'impactor_id', 'm_target', 'm_impactor',
    'Mass', 'Impact ratio', 'events', 'P_equil', 'T_equil', 'k_mantle',
}
for key, event_table in data_dict.items():
    missing = required_event_columns - set(event_table.columns)
    if missing:
        raise ValueError(f'{key} lacks required event columns: {sorted(missing)}')
    if event_table[['target_id', 'impactor_id']].isna().any().any():
        raise ValueError(f'{key} contains missing target or impactor IDs.')
    unexpected = set(event_table['events'].dropna().unique()) - {'small', 'global'}
    if unexpected:
        raise ValueError(f'{key} contains unexpected event types: {sorted(unexpected)}')
    pt = event_table[['P_equil', 'T_equil']].to_numpy(dtype=float)
    if not np.isfinite(pt).all() or (pt <= 0.0).any():
        raise ValueError(f'{key} contains invalid equilibration P/T values.')
    melting = event_table['events'].eq('global')
    fractions = event_table.loc[melting, 'k_mantle'].to_numpy(dtype=float)
    if not np.isfinite(fractions).all() or ((fractions < 0.0) | (fractions > 1.0)).any():
        raise ValueError(f'{key} contains invalid global melt fractions.')

save_dir = workflow_dir / 'Tables'
save_dir.mkdir(parents=True, exist_ok=True)
save_file = Path(OUTPUT_EVENTS_FILE) if OUTPUT_EVENTS_FILE is not None else save_dir / '05_small_global_impact_events_S.xlsx'
with pd.ExcelWriter(save_file) as writer:
    for key, event_table in data_dict.items():
        event_table.to_excel(writer, sheet_name=key, index=False)

exported_events = pd.read_excel(save_file, sheet_name=None)
if list(exported_events) != list(data_dict):
    raise AssertionError('Exported planet sheets changed.')
for key in data_dict:
    if len(exported_events[key]) != len(data_dict[key]):
        raise AssertionError(f'Exported row count changed for {key}.')
parameter_file = Path(OUTPUT_PARAMETERS_FILE) if OUTPUT_PARAMETERS_FILE is not None else save_dir / '05_impact_parameters_S.json'
parameter_file.write_text(json.dumps({
    'P_RATIO': P_RATIO,
    'GLOBAL_MASS_THRESHOLD': GLOBAL_MASS_THRESHOLD,
    'IMPACT_RATIO_RANGE': list(IMPACT_RATIO_RANGE),
    'COOLING_DURATION_MYR': COOLING_DURATION_MYR,
}, indent=2) + '\n')
print(f'Saved intermediate event table: {save_file}')
print(f'Saved impact parameters: {parameter_file}')


Saved intermediate event table: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S/Tables/05_small_global_impact_events_S.xlsx
Saved impact parameters: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S/Tables/05_impact_parameters_S.json
